In [1]:
# Force install by calling the files directly and ignoring compatibility tags
!pip install /kaggle/input/datasets/pjleek/onnxruntime126-312/onnxruntime-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl --no-deps --force-reinstall


Processing /kaggle/input/datasets/pjleek/onnxruntime126-312/onnxruntime-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl


In [2]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.8 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import glob
import math
import numpy as np
import os
from collections import namedtuple

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])
CENTER_X, CENTER_Y = 50.0, 50.0
MAX_SPEED = 6.0

# User's Discovered Top-10 Deadliest Coordinates (The Danger Grid)
TRAPS = [(15, 83), (5, 67), (9, 76), (75, 75), (89, 61), (6, 28), (90, 68), (95, 69), (56, 9), (36, 18)]

# ============================================================
# 1. Feature Engineering Helpers
# ============================================================
def get_danger_heat(tx, ty):
    min_d = min(math.hypot(tx - t[0], ty - t[1]) for t in TRAPS)
    return max(0.0, 1.0 - (min_d / 15.0)) # 1.0 if right on the trap, fading to 0.0

def min_mirror_dist(tx, ty, sx, sy):
    # Calculates distance to the 3 mirrored variants of the source
    d1 = math.hypot(tx - (100.0 - sx), ty - sy)
    d2 = math.hypot(tx - sx, ty - (100.0 - sy))
    d3 = math.hypot(tx - (100.0 - sx), ty - (100.0 - sy))
    return min(d1, d2, d3)

# ============================================================
# 2. OrbitNet V5 (Symmetry & Heatmap Aware)
# ============================================================
class OrbitNet(nn.Module):
    def __init__(self):
        super(OrbitNet, self).__init__()
        self.fc1 = nn.Linear(12, 64) # 12 Engineered Features
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU() # Swapped from LeakyReLU to fix ONNX conversion

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.sigmoid(self.fc3(x))
        
    def sigmoid(self, x):
        return torch.sigmoid(x)

# ============================================================
# 3. Crash-Proof Physics (No 'initial_planets' reliance)
# ============================================================
def fleet_speed(ships):
    if ships <= 1: return 1.0
    return 1.0 + (MAX_SPEED - 1.0) * (min(1.0, math.log(ships) / math.log(1000.0)) ** 1.5)

def get_target_pos(tgt, turns, ang_vel, comets, comet_ids):
    if tgt.id in comet_ids:
        for c in comets:
            if tgt.id in c.get("planet_ids",[]):
                idx = c["planet_ids"].index(tgt.id)
                f_idx = c.get("path_index", 0) + turns
                if idx < len(c["paths"]) and 0 <= f_idx < len(c["paths"][idx]): 
                    return c["paths"][idx][f_idx]
        return (tgt.x, tgt.y)
    
    # Orbital Leading direct from current state
    if ang_vel == 0.0: return (tgt.x, tgt.y)
    r = math.hypot(tgt.x - CENTER_X, tgt.y - CENTER_Y)
    ang = math.atan2(tgt.y - CENTER_Y, tgt.x - CENTER_X) + ang_vel * turns
    return (CENTER_X + r * math.cos(ang), CENTER_Y + r * math.sin(ang))

def plan_flight(src, tgt, ships, ang_vel, comets, comet_ids):
    speed = fleet_speed(ships)
    tx, ty = tgt.x, tgt.y
    eta = 0.0
    for _ in range(5): 
        flight_dist = max(0.0, math.hypot(tx - src.x, ty - src.y) - src.radius - 0.1 - tgt.radius)
        eta = flight_dist / speed
        tx, ty = get_target_pos(tgt, int(math.ceil(eta)), ang_vel, comets, comet_ids)
    
    angle = math.atan2(ty - src.y, tx - src.x)
    return angle, int(math.ceil(eta)), tx, ty, speed

# ============================================================
# 4. Data Extraction & Training
# ============================================================
def extract_training_chunk(files_chunk):
    X_data, Y_data =[],[]
    for filepath in files_chunk:
        try:
            with open(filepath, 'r') as f: match = json.load(f)
            steps = match.get("steps",[])
            if not steps: continue
            
            winner_id = [a.get("reward", -1) for a in steps[-1]].index(max([a.get("reward", -1) for a in steps[-1]]))
            
            for step_idx, step in enumerate(steps[:-25]): 
                obs = step[0].get("observation", {})
                planets = [Planet(*p) for p in obs.get("planets", [])]
                ang_vel = obs.get("angular_velocity", 0.0)
                comets = obs.get("comets",[])
                comet_ids = set(obs.get("comet_planet_ids",[]))
                
                for player_id in range(len(step)):
                    actions = step[player_id].get("action",[])
                    is_winner = (player_id == winner_id)
                    
                    for act in actions:
                        src_id, angle, ships_sent = act[0], act[1], max(1, act[2])
                        src_p = next((p for p in planets if p.id == src_id), None)
                        if not src_p: continue

                        best_diff, tgt_p = float('inf'), None
                        for p in planets:
                            if p.id == src_id: continue
                            tgt_ang = math.atan2(p.y - src_p.y, p.x - src_p.x)
                            ang_diff = abs((angle - tgt_ang + math.pi) % (2 * math.pi) - math.pi)
                            if ang_diff < best_diff: best_diff, tgt_p = ang_diff, p
                                
                        if best_diff > 0.2 or not tgt_p: continue 
                            
                        angle_res, eta, tx, ty, speed = plan_flight(src_p, tgt_p, ships_sent, ang_vel, comets, comet_ids)
                        if eta > 99: continue
                        
                        # --- THE 12 ENGINEERED FEATURES ---
                        feat =[
                            (step_idx // 50) / 10.0,                                    # 1. Turn_Group
                            abs(tx - 50.0) / 50.0,                                      # 2. Folded_X
                            abs(ty - 50.0) / 50.0,                                      # 3. Folded_Y
                            tgt_p.production / 5.0,                                     # 4. Target_Prod
                            1.0 if ships_sent >= 10 else 0.0,                           # 5. Is_Urgent
                            min_mirror_dist(tx, ty, src_p.x, src_p.y) / 100.0,          # 6. Mirror_Dist
                            get_danger_heat(tx, ty),                                    # 7. Danger_Heat
                            min(1.0, tgt_p.production / max(1.0, float(ships_sent))),   # 8. ROI_Efficiency
                            eta / 50.0,                                                 # 9. Normalized_ETA
                            ships_sent / max(1.0, float(src_p.ships)),                  # 10. Capital_Risk
                            1.0 if tgt_p.owner == player_id else 0.0,                   # 11. Is_Defensive
                            1.0 if is_winner else 0.0                                   # 12. Success Context
                        ]
                        
                        # Apply heavy penalty to moves leading into Danger Grids
                        score = 0.5 if is_winner else 0.0
                        if get_danger_heat(tx, ty) > 0.5 and tgt_p.owner != player_id:
                            score *= 0.1 
                            
                        X_data.append(feat)
                        Y_data.append([score])
                        
        except Exception: continue
    
    if not X_data: return torch.empty(0, 12), torch.empty(0, 1)
    return torch.tensor(X_data, dtype=torch.float32), torch.tensor(Y_data, dtype=torch.float32)

def train_and_export():
    dataset_path = "/kaggle/input/datasets/bovard/orbit-wars-top10-episodes-2026-05-04/episodes/episodes"
    files = glob.glob(os.path.join(dataset_path, "**/*.json"), recursive=True)
    if not files: return print("No training data found!")
        
    model = OrbitNet()
    optimizer = optim.Adam(model.parameters(), lr=0.001) 
    chunk_size = 500
    
    for i in range(0, len(files), chunk_size):
        chunk_files = files[i:i+chunk_size]
        X_train, Y_train = extract_training_chunk(chunk_files)
        if len(X_train) == 0: continue
            
        dataset = TensorDataset(X_train, Y_train)
        dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)
        
        for epoch in range(5): 
            for batch_X, batch_Y in dataloader:
                optimizer.zero_grad()
                outputs = model(batch_X)
                weights = torch.where(batch_X[:, 11:12] == 0.0, 2.0, 1.0)
                loss = (weights * (outputs - batch_Y) ** 2).mean()
                loss.backward()
                optimizer.step()
                
    model.eval() 
    dummy_input = torch.randn(1, 12)
    export_path = "/kaggle/working/orbit_model.onnx"
    import onnx
    # Updated to opset 18 to fix Kaggle PyTorch fallback error
    torch.onnx.export(model, dummy_input, export_path, export_params=True, opset_version=18, do_constant_folding=True, input_names=['input'], output_names=['output'])
    onnx_model = onnx.load(export_path)
    onnx_model.ir_version = 9
    onnx.save(onnx_model, export_path)
    print(f"Saved OrbitNet V5 to: {export_path}")

if __name__ == "__main__":
    train_and_export()

[torch.onnx] Obtain model graph for `OrbitNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `OrbitNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Saved OrbitNet V5 to: /kaggle/working/orbit_model.onnx


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
